# 02 - Pré-processamento de Dados do ENEM

Este notebook realiza o pré-processamento dos microdados do ENEM 2024.

**Arquivo:** `RESULTADOS_2024.csv`

In [32]:
# Imports
import pandas as pd
import numpy as np
import warnings
import sys
import pickle
from pathlib import Path
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

from src.data_prep import (
    carregar_dados_enem,
    limpar_dados_enem,
    agregar_por_escola,
    criar_features_adicionais
)

print('✓ Imports realizados')

✓ Imports realizados


In [33]:
# Configuracoes
DATA_RAW = project_root / '..' / 'microdados_enem_2024' / 'DADOS' / 'RESULTADOS_2024.csv'
DATA_PROCESSED = project_root / 'data' / 'processed'
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

FRAC_AMOSTRA = None  # None = 100% dos dados (4.3M linhas)
MIN_ALUNOS = 10

print('Caminho:', DATA_RAW)
if FRAC_AMOSTRA:
    print('Amostra:', FRAC_AMOSTRA * 100, '%')
else:
    print('Amostra: 100% (todos os dados - 4.3M linhas)')

Caminho: C:\Users\leofs\Documents\ciencia_de_dados_2\enem_clustering\..\microdados_enem_2024\DADOS\RESULTADOS_2024.csv
Amostra: 100% (todos os dados - 4.3M linhas)


In [34]:
%%time
df_raw = carregar_dados_enem(
    caminho_arquivo=str(DATA_RAW),
    amostra=FRAC_AMOSTRA
)
print('Dados:', df_raw.shape)
print('Colunas:', df_raw.columns.tolist())

Carregando dados de: C:\Users\leofs\Documents\ciencia_de_dados_2\enem_clustering\..\microdados_enem_2024\DADOS\RESULTADOS_2024.csv
Dados carregados: 4332944 linhas, 42 colunas
Dados: (4332944, 42)
Colunas: ['NU_SEQUENCIAL', 'NU_ANO', 'CO_ESCOLA', 'CO_MUNICIPIO_ESC', 'NO_MUNICIPIO_ESC', 'CO_UF_ESC', 'SG_UF_ESC', 'TP_DEPENDENCIA_ADM_ESC', 'TP_LOCALIZACAO_ESC', 'TP_SIT_FUNC_ESC', 'CO_MUNICIPIO_PROVA', 'NO_MUNICIPIO_PROVA', 'CO_UF_PROVA', 'SG_UF_PROVA', 'TP_PRESENCA_CN', 'TP_PRESENCA_CH', 'TP_PRESENCA_LC', 'TP_PRESENCA_MT', 'CO_PROVA_CN', 'CO_PROVA_CH', 'CO_PROVA_LC', 'CO_PROVA_MT', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'TX_RESPOSTAS_CN', 'TX_RESPOSTAS_CH', 'TX_RESPOSTAS_LC', 'TX_RESPOSTAS_MT', 'TP_LINGUA', 'TX_GABARITO_CN', 'TX_GABARITO_CH', 'TX_GABARITO_LC', 'TX_GABARITO_MT', 'TP_STATUS_REDACAO', 'NU_NOTA_COMP1', 'NU_NOTA_COMP2', 'NU_NOTA_COMP3', 'NU_NOTA_COMP4', 'NU_NOTA_COMP5', 'NU_NOTA_REDACAO']
CPU times: total: 36.3 s
Wall time: 36.9 s


In [35]:
# Selecionar colunas - incluindo tipo de dependencia administrativa
COLUNAS_NOTAS = ['NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO']
COLUNAS_ESCOLA = ['CO_ESCOLA', 'CO_MUNICIPIO_ESC', 'CO_UF_ESC', 'TP_DEPENDENCIA_ADM_ESC', 'TP_LOCALIZACAO_ESC']
COLUNAS_PRESENCA = ['TP_PRESENCA_CN', 'TP_PRESENCA_CH', 'TP_PRESENCA_LC', 'TP_PRESENCA_MT']

colunas_desejadas = COLUNAS_NOTAS + COLUNAS_ESCOLA + COLUNAS_PRESENCA
colunas_existentes = [c for c in colunas_desejadas if c in df_raw.columns]
colunas_faltantes = [c for c in colunas_desejadas if c not in df_raw.columns]

print(f'Colunas encontradas: {len(colunas_existentes)}')
if colunas_faltantes:
    print(f'Colunas faltantes: {colunas_faltantes}')
print('Colunas:', colunas_existentes)

df = df_raw[colunas_existentes].copy()

Colunas encontradas: 14
Colunas: ['NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO', 'CO_ESCOLA', 'CO_MUNICIPIO_ESC', 'CO_UF_ESC', 'TP_DEPENDENCIA_ADM_ESC', 'TP_LOCALIZACAO_ESC', 'TP_PRESENCA_CN', 'TP_PRESENCA_CH', 'TP_PRESENCA_LC', 'TP_PRESENCA_MT']


In [36]:
# Limpeza
coluna_escola = 'CO_ESCOLA' if 'CO_ESCOLA' in df.columns else 'CO_MUNICIPIO_ESC'
df = df.dropna(subset=[coluna_escola])
df = df[(df['TP_PRESENCA_CN']==1) & (df['TP_PRESENCA_CH']==1) & (df['TP_PRESENCA_LC']==1) & (df['TP_PRESENCA_MT']==1)]
print('Apos limpeza:', len(df), 'registros')
print('Escolas unicas:', df[coluna_escola].nunique())

Apos limpeza: 1193432 registros
Escolas unicas: 28901


In [37]:
# Agregar por escola
# Incluir colunas categóricas para análise posterior
colunas_categoricas = ['TP_DEPENDENCIA_ADM_ESC', 'CO_UF_ESC', 'TP_LOCALIZACAO_ESC']
colunas_categoricas = [c for c in colunas_categoricas if c in df.columns]

df_escolas = agregar_por_escola(
    df, 
    coluna_escola=coluna_escola, 
    colunas_notas=[c for c in COLUNAS_NOTAS if c in df.columns],
    colunas_categoricas=colunas_categoricas
)
print('Escolas:', len(df_escolas))
print('Colunas:', df_escolas.columns.tolist())


Agregando por escola:
- Coluna escola: CO_ESCOLA
- Colunas de notas: ['NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO']
- Colunas categóricas: ['TP_DEPENDENCIA_ADM_ESC', 'CO_UF_ESC', 'TP_LOCALIZACAO_ESC']
- Colunas numéricas adicionais: []
- Colunas apos agg (antes de renomear): ['CO_ESCOLA', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO', 'TP_DEPENDENCIA_ADM_ESC', 'CO_UF_ESC', 'TP_LOCALIZACAO_ESC', 'QTD_ALUNOS']
- Colunas finais: ['CO_ESCOLA', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO', 'TP_DEPENDENCIA_ADM_ESC', 'CO_UF_ESC', 'TP_LOCALIZACAO_ESC', 'QTD_ALUNOS']
- Escolas agregadas: 28901
Escolas: 28901
Colunas: ['CO_ESCOLA', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO', 'TP_DEPENDENCIA_ADM_ESC', 'CO_UF_ESC', 'TP_LOCALIZACAO_ESC', 'QTD_ALUNOS']


In [38]:
# Limpar escolas pequenas
df_escolas_limpo = limpar_dados_enem(df_escolas, min_alunos_por_escola=MIN_ALUNOS)
print('Apos limpeza:', len(df_escolas_limpo), 'escolas')


Limpeza de dados:
- Linhas originais: 28901
- Após remover sem escola: 28901
- Após remover notas ausentes: 28901
- Após filtrar escolas (QTD_ALUNOS >= 10): 23023
Apos limpeza: 23023 escolas


In [39]:
# Criar features adicionais
df_features = criar_features_adicionais(df_escolas_limpo)
print('Colunas em df_features:')
print(df_features.columns.tolist())


Criando features adicionais:
- Colunas de entrada: ['CO_ESCOLA', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO', 'TP_DEPENDENCIA_ADM_ESC', 'CO_UF_ESC', 'TP_LOCALIZACAO_ESC', 'QTD_ALUNOS']
- Colunas de notas encontradas: ['NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO']
- MEDIA_GERAL criada
- STD_NOTAS criada
- NOTA_MAX e NOTA_MIN criadas
- DIF_MT_LC criada
- DIF_CN_CH criada
- PCT_NU_NOTA_CN criada
- PCT_NU_NOTA_CH criada
- PCT_NU_NOTA_LC criada
- PCT_NU_NOTA_MT criada
- PCT_NU_NOTA_REDACAO criada
- Colunas de saída: ['CO_ESCOLA', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO', 'TP_DEPENDENCIA_ADM_ESC', 'CO_UF_ESC', 'TP_LOCALIZACAO_ESC', 'QTD_ALUNOS', 'MEDIA_GERAL', 'STD_NOTAS', 'NOTA_MAX', 'NOTA_MIN', 'DIF_MT_LC', 'DIF_CN_CH', 'PCT_NU_NOTA_CN', 'PCT_NU_NOTA_CH', 'PCT_NU_NOTA_LC', 'PCT_NU_NOTA_MT', 'PCT_NU_NOTA_REDACAO']
Colunas em df_features:
['CO_ESCOLA', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NO

In [40]:
# Preparar para clustering - VERIFICACAO COMPLETA
print('=== VERIFICACAO DE df_features ===')
print('Shape df_features:', df_features.shape)
print('Index df_features:', type(df_features.index), len(df_features.index))
print('Index unico?', df_features.index.is_unique)

# Verificar se df_features tem linhas
if len(df_features) == 0:
    print('ERRO CRITICO: df_features esta vazio!')
else:
    print(f'df_features tem {len(df_features)} linhas')
    print('\nPrimeiras 3 linhas:')
    print(df_features.head(3))

print('\n=== VERIFICACAO DE COLUNAS ===')
print('Colunas disponiveis:', df_features.columns.tolist())

# Identificar features numericas
COLUNAS_NOTAS = ['NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO']
features_possiveis = ['MEDIA_GERAL', 'STD_NOTAS'] + COLUNAS_NOTAS
features_numericas = [c for c in features_possiveis if c in df_features.columns]

print('\nFeatures selecionadas:', features_numericas)

if len(features_numericas) == 0:
    print('ERRO: Nenhuma feature numerica encontrada!')
elif len(df_features) == 0:
    print('ERRO: df_features esta vazio!')
else:
    # Extrair X
    X = df_features[features_numericas]
    print(f'\nX tipo: {type(X)}')
    print(f'X shape: {X.shape}')
    print(f'X vazio? {X.empty}')
    
    # Verificar valores nulos
    print('\nValores nulos por coluna em X:')
    print(X.isnull().sum())
    
    # Forçar conversao para numerico
    print('\nConvertendo para numerico...')
    X = X.apply(pd.to_numeric, errors='coerce')
    print('Apos conversao:')
    print('X shape:', X.shape)
    print('Valores nulos:', X.isnull().sum().sum())
    
    # Remover NaN
    X_clean = X.dropna()
    print(f'\nX apos dropna: {X_clean.shape}')
    
    if X_clean.empty:
        print('ERRO: X ainda esta vazio! Verificando dados originais...')
        print('\nValores das notas em df_features:')
        for col in COLUNAS_NOTAS:
            if col in df_features.columns:
                print(f'{col}: {df_features[col].dtype}, valores unicos: {df_features[col].nunique()}')
                print(f'  Primeiros valores: {df_features[col].head(3).tolist()}')
    else:
        X = X_clean
        df_final = df_features.loc[X.index].copy()
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        print(f'\n✓ Dataset final: {X.shape}')
        print('Media:', round(X_scaled.mean(), 4))
        print('Std:', round(X_scaled.std(), 4))

=== VERIFICACAO DE df_features ===
Shape df_features: (23023, 21)
Index df_features: <class 'pandas.RangeIndex'> 23023
Index unico? True
df_features tem 23023 linhas

Primeiras 3 linhas:
    CO_ESCOLA  NU_NOTA_CN  NU_NOTA_CH  NU_NOTA_LC  NU_NOTA_MT  \
0  11000058.0  573.838462  587.475824  580.107692  643.553846   
1  11000198.0  508.665957  516.668085  541.334043  526.182979   
2  11000252.0  535.600000  568.070588  576.505882  605.447059   

   NU_NOTA_REDACAO  TP_DEPENDENCIA_ADM_ESC  CO_UF_ESC  TP_LOCALIZACAO_ESC  \
0       785.494505                     4.0       11.0                 1.0   
1       717.021277                     4.0       11.0                 1.0   
2       717.647059                     4.0       11.0                 1.0   

   QTD_ALUNOS  ...  STD_NOTAS    NOTA_MAX    NOTA_MIN  DIF_MT_LC  DIF_CN_CH  \
0          91  ...  89.064734  785.494505  573.838462  63.446154 -13.637363   
1          47  ...  87.522906  717.021277  508.665957 -15.151064  -8.002128   
2     

In [41]:
# Salvar dados processados (somente se X_scaled existir)
if 'X_scaled' in locals() and 'df_final' in locals():
    df_final.to_parquet(DATA_PROCESSED / 'dados_escolas.parquet')
    
    df_scaled = pd.DataFrame(X_scaled, columns=features_numericas, index=df_final.index)
    df_scaled.to_parquet(DATA_PROCESSED / 'dados_clustering.parquet')
    
    with open(DATA_PROCESSED / 'scaler.pkl', 'wb') as f:
        pickle.dump(scaler, f)
    
    with open(DATA_PROCESSED / 'features.txt', 'w') as f:
        f.write('\n'.join(features_numericas))
    
    print('Dados salvos com sucesso!')
    print('Arquivos em:', DATA_PROCESSED)
else:
    print('Dados nao foram salvos. Verifique os erros anteriores.')

Dados salvos com sucesso!
Arquivos em: C:\Users\leofs\Documents\ciencia_de_dados_2\enem_clustering\data\processed
